# 📈 Tasks 13-16: Time Series Analysis & Recommender System

---

## 🎯 Objectives
1. Build time series forecasting for battery SoC prediction
2. Implement real-time anomaly detection
3. Create content-based mission recommender system
4. Implement similarity metrics for mission matching

---

## 🔬 Mathematical Foundation

### Part A: Time Series Analysis

#### 1. Moving Average (MA)

Simple Moving Average (SMA):
$$SMA_t = \frac{1}{n} \sum_{i=0}^{n-1} x_{t-i}$$

Exponential Moving Average (EMA):
$$EMA_t = \alpha \cdot x_t + (1-\alpha) \cdot EMA_{t-1}$$

where $\alpha = \frac{2}{n+1}$ (smoothing factor)

---

#### 2. Linear Trend Forecasting

Fit a line to time series:
$$\hat{y}_t = a + b \cdot t$$

where $b = \frac{\sum(t - \bar{t})(y - \bar{y})}{\sum(t - \bar{t})^2}$

---

#### 3. Anomaly Detection (Z-Score Method)

$$z = \frac{x - \mu}{\sigma}$$

Anomaly if $|z| > threshold$ (typically 2-3)

---

### Part B: Recommender System

#### 4. Cosine Similarity

$$\cos(\theta) = \frac{A \cdot B}{\|A\| \|B\|} = \frac{\sum a_i b_i}{\sqrt{\sum a_i^2} \sqrt{\sum b_i^2}}$$

Range: [-1, 1] or [0, 1] for non-negative vectors

---

#### 5. Euclidean Distance

$$d(A, B) = \sqrt{\sum_{i=1}^{n}(a_i - b_i)^2}$$

Similarity: $sim = \frac{1}{1 + d}$

---

## 📦 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Libraries loaded!")

In [ ]:
# Load data
df = pd.read_csv('../data/raw/uav_telemetry.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"Dataset: {len(df)} samples")
print(f"Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")

---

# 📊 PART A: Time Series Analysis (Tasks 13-14)

---

## 🔧 Time Series Functions (From Scratch)

In [ ]:
class TimeSeriesAnalyzer:
    """
    Time series analysis tools implemented from scratch.
    """
    
    @staticmethod
    def simple_moving_average(data, window):
        """
        Calculate Simple Moving Average.
        SMA_t = (1/n) * Σ x_{t-i}
        """
        data = np.array(data)
        n = len(data)
        sma = np.full(n, np.nan)
        
        for i in range(window - 1, n):
            sma[i] = np.mean(data[i - window + 1:i + 1])
        
        return sma
    
    @staticmethod
    def exponential_moving_average(data, span):
        """
        Calculate Exponential Moving Average.
        EMA_t = α * x_t + (1-α) * EMA_{t-1}
        """
        data = np.array(data)
        alpha = 2 / (span + 1)
        ema = np.zeros(len(data))
        ema[0] = data[0]
        
        for i in range(1, len(data)):
            ema[i] = alpha * data[i] + (1 - alpha) * ema[i-1]
        
        return ema
    
    @staticmethod
    def linear_trend(data):
        """
        Fit linear trend: y = a + b*t
        Returns: slope, intercept, trend_line
        """
        data = np.array(data)
        n = len(data)
        t = np.arange(n)
        
        t_mean = np.mean(t)
        y_mean = np.mean(data)
        
        slope = np.sum((t - t_mean) * (data - y_mean)) / np.sum((t - t_mean) ** 2)
        intercept = y_mean - slope * t_mean
        
        trend_line = intercept + slope * t
        
        return slope, intercept, trend_line
    
    @staticmethod
    def forecast_next_n(data, n_steps, method='ema', span=10):
        """
        Forecast next n steps using trend + EMA.
        """
        data = np.array(data)
        slope, intercept, _ = TimeSeriesAnalyzer.linear_trend(data)
        ema = TimeSeriesAnalyzer.exponential_moving_average(data, span)
        
        # Combine trend with recent EMA
        last_idx = len(data) - 1
        forecasts = []
        
        for i in range(1, n_steps + 1):
            # Trend component
            trend = slope * (last_idx + i)
            # Level from EMA
            level = ema[-1]
            # Combined forecast
            forecast = level + slope * i
            forecasts.append(forecast)
        
        return np.array(forecasts)


print("✅ TimeSeriesAnalyzer class defined!")

In [ ]:
# Analyze battery SoC time series
soc_series = df['battery_soc'].values[:500]  # Use first 500 for visualization

# Calculate moving averages
sma_10 = TimeSeriesAnalyzer.simple_moving_average(soc_series, 10)
sma_50 = TimeSeriesAnalyzer.simple_moving_average(soc_series, 50)
ema_10 = TimeSeriesAnalyzer.exponential_moving_average(soc_series, 10)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Raw SoC with Moving Averages
ax = axes[0, 0]
ax.plot(soc_series, 'lightgray', alpha=0.7, label='Raw SoC')
ax.plot(sma_10, 'blue', linewidth=2, label='SMA(10)')
ax.plot(sma_50, 'red', linewidth=2, label='SMA(50)')
ax.set_xlabel('Time Step')
ax.set_ylabel('Battery SoC (%)')
ax.set_title('Battery SoC with Moving Averages', fontweight='bold')
ax.legend()

# 2. SMA vs EMA comparison
ax = axes[0, 1]
ax.plot(soc_series, 'lightgray', alpha=0.7, label='Raw')
ax.plot(sma_10, 'blue', linewidth=2, label='SMA(10)')
ax.plot(ema_10, 'green', linewidth=2, label='EMA(10)')
ax.set_xlabel('Time Step')
ax.set_ylabel('Battery SoC (%)')
ax.set_title('SMA vs EMA (EMA reacts faster)', fontweight='bold')
ax.legend()

# 3. Trend Analysis
slope, intercept, trend = TimeSeriesAnalyzer.linear_trend(soc_series)
ax = axes[1, 0]
ax.scatter(range(len(soc_series)), soc_series, alpha=0.3, s=5, label='Data')
ax.plot(trend, 'r-', linewidth=2, label=f'Trend (slope={slope:.4f})')
ax.set_xlabel('Time Step')
ax.set_ylabel('Battery SoC (%)')
ax.set_title('Linear Trend Analysis', fontweight='bold')
ax.legend()

# 4. Forecasting
train_data = soc_series[:400]
test_data = soc_series[400:]
forecasts = TimeSeriesAnalyzer.forecast_next_n(train_data, len(test_data))

ax = axes[1, 1]
ax.plot(range(400), train_data, 'blue', label='Training Data')
ax.plot(range(400, 500), test_data, 'green', label='Actual')
ax.plot(range(400, 500), forecasts, 'red', linestyle='--', label='Forecast')
ax.axvline(x=400, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('Time Step')
ax.set_ylabel('Battery SoC (%)')
ax.set_title('Time Series Forecasting', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/time_series_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 🚨 Task 14: Anomaly Detection

In [ ]:
class AnomalyDetector:
    """
    Anomaly detection methods from scratch.
    """
    
    @staticmethod
    def zscore_detection(data, threshold=3.0):
        """
        Detect anomalies using Z-score method.
        Anomaly if |z| > threshold
        """
        data = np.array(data)
        mean = np.mean(data)
        std = np.std(data)
        
        z_scores = (data - mean) / std
        anomalies = np.abs(z_scores) > threshold
        
        return anomalies, z_scores
    
    @staticmethod
    def iqr_detection(data, k=1.5):
        """
        Detect anomalies using IQR method.
        Anomaly if x < Q1 - k*IQR or x > Q3 + k*IQR
        """
        data = np.array(data)
        q1 = np.percentile(data, 25)
        q3 = np.percentile(data, 75)
        iqr = q3 - q1
        
        lower = q1 - k * iqr
        upper = q3 + k * iqr
        
        anomalies = (data < lower) | (data > upper)
        
        return anomalies, (lower, upper)
    
    @staticmethod
    def rolling_zscore(data, window=20, threshold=2.5):
        """
        Detect anomalies using rolling window Z-score.
        Better for time series with changing distributions.
        """
        data = np.array(data)
        n = len(data)
        anomalies = np.zeros(n, dtype=bool)
        z_scores = np.zeros(n)
        
        for i in range(window, n):
            window_data = data[i-window:i]
            mean = np.mean(window_data)
            std = np.std(window_data)
            
            if std > 0:
                z = (data[i] - mean) / std
                z_scores[i] = z
                anomalies[i] = abs(z) > threshold
        
        return anomalies, z_scores


print("✅ AnomalyDetector class defined!")

In [ ]:
# Apply anomaly detection to power consumption
power_series = df['power_consumption'].values

# Detect anomalies
zscore_anom, z_scores = AnomalyDetector.zscore_detection(power_series, threshold=3.0)
iqr_anom, bounds = AnomalyDetector.iqr_detection(power_series, k=1.5)
rolling_anom, rolling_z = AnomalyDetector.rolling_zscore(power_series, window=50, threshold=2.5)

print(f"Z-Score Anomalies: {zscore_anom.sum()} ({zscore_anom.mean():.2%})")
print(f"IQR Anomalies: {iqr_anom.sum()} ({iqr_anom.mean():.2%})")
print(f"Rolling Z-Score Anomalies: {rolling_anom.sum()} ({rolling_anom.mean():.2%})")

In [ ]:
# Visualize anomaly detection
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Use subset for visualization
idx = 500
x = np.arange(idx)

# 1. Z-Score detection
ax = axes[0, 0]
ax.plot(x, power_series[:idx], 'blue', alpha=0.7)
ax.scatter(x[zscore_anom[:idx]], power_series[:idx][zscore_anom[:idx]], 
           c='red', s=50, label='Anomalies', zorder=5)
ax.set_xlabel('Time Step')
ax.set_ylabel('Power (W)')
ax.set_title(f'Z-Score Anomaly Detection (threshold=3)', fontweight='bold')
ax.legend()

# 2. IQR detection
ax = axes[0, 1]
ax.plot(x, power_series[:idx], 'blue', alpha=0.7)
ax.axhline(y=bounds[0], color='green', linestyle='--', label=f'Lower: {bounds[0]:.0f}')
ax.axhline(y=bounds[1], color='red', linestyle='--', label=f'Upper: {bounds[1]:.0f}')
ax.scatter(x[iqr_anom[:idx]], power_series[:idx][iqr_anom[:idx]], 
           c='red', s=50, zorder=5)
ax.set_xlabel('Time Step')
ax.set_ylabel('Power (W)')
ax.set_title('IQR Anomaly Detection', fontweight='bold')
ax.legend()

# 3. Z-Score distribution
ax = axes[1, 0]
ax.hist(z_scores, bins=50, color='steelblue', edgecolor='white')
ax.axvline(x=-3, color='red', linestyle='--', label='Threshold (-3)')
ax.axvline(x=3, color='red', linestyle='--', label='Threshold (+3)')
ax.set_xlabel('Z-Score')
ax.set_ylabel('Frequency')
ax.set_title('Z-Score Distribution', fontweight='bold')
ax.legend()

# 4. Rolling Z-Score
ax = axes[1, 1]
ax.plot(x, rolling_z[:idx], 'purple', alpha=0.7)
ax.axhline(y=2.5, color='red', linestyle='--')
ax.axhline(y=-2.5, color='red', linestyle='--')
ax.fill_between(x, -2.5, 2.5, alpha=0.1, color='green')
ax.set_xlabel('Time Step')
ax.set_ylabel('Rolling Z-Score')
ax.set_title('Rolling Z-Score (window=50)', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/anomaly_detection.png', dpi=150, bbox_inches='tight')
plt.show()

---

# 🎯 PART B: Content-Based Recommender System (Tasks 15-16)

---

In [ ]:
class SimilarityMetrics:
    """
    Similarity metrics for content-based recommendations.
    """
    
    @staticmethod
    def cosine_similarity(a, b):
        """
        Cosine similarity: cos(θ) = (A·B) / (||A|| ||B||)
        """
        a, b = np.array(a), np.array(b)
        dot_product = np.dot(a, b)
        norm_a = np.linalg.norm(a)
        norm_b = np.linalg.norm(b)
        
        if norm_a == 0 or norm_b == 0:
            return 0
        
        return dot_product / (norm_a * norm_b)
    
    @staticmethod
    def euclidean_similarity(a, b):
        """
        Euclidean similarity: 1 / (1 + distance)
        """
        a, b = np.array(a), np.array(b)
        distance = np.sqrt(np.sum((a - b) ** 2))
        return 1 / (1 + distance)
    
    @staticmethod
    def manhattan_similarity(a, b):
        """
        Manhattan similarity: 1 / (1 + |distance|)
        """
        a, b = np.array(a), np.array(b)
        distance = np.sum(np.abs(a - b))
        return 1 / (1 + distance)
    
    @staticmethod
    def pearson_similarity(a, b):
        """
        Pearson correlation as similarity.
        """
        a, b = np.array(a), np.array(b)
        a_mean, b_mean = np.mean(a), np.mean(b)
        
        numerator = np.sum((a - a_mean) * (b - b_mean))
        denominator = np.sqrt(np.sum((a - a_mean)**2) * np.sum((b - b_mean)**2))
        
        if denominator == 0:
            return 0
        
        return numerator / denominator


print("✅ SimilarityMetrics class defined!")

In [ ]:
class MissionRecommender:
    """
    Content-based mission recommender system.
    
    Recommends missions based on:
    - Current UAV state (battery, conditions)
    - Historical successful missions
    - Similarity matching
    """
    
    def __init__(self, similarity_metric='cosine'):
        self.similarity_metric = similarity_metric
        self.mission_database = None
        self.feature_columns = None
        self.scaler_mean = None
        self.scaler_std = None
    
    def _get_similarity_func(self):
        """Get similarity function based on metric name."""
        metrics = {
            'cosine': SimilarityMetrics.cosine_similarity,
            'euclidean': SimilarityMetrics.euclidean_similarity,
            'manhattan': SimilarityMetrics.manhattan_similarity,
            'pearson': SimilarityMetrics.pearson_similarity
        }
        return metrics.get(self.similarity_metric, SimilarityMetrics.cosine_similarity)
    
    def fit(self, df, feature_columns):
        """
        Build mission database from successful missions.
        """
        self.feature_columns = feature_columns
        
        # Filter successful missions
        successful = df[df['mission_success'] == 1].copy()
        
        # Normalize features
        self.scaler_mean = successful[feature_columns].mean().values
        self.scaler_std = successful[feature_columns].std().values
        self.scaler_std[self.scaler_std == 0] = 1
        
        # Store normalized features
        self.mission_database = successful.copy()
        normalized = (successful[feature_columns].values - self.scaler_mean) / self.scaler_std
        
        for i, col in enumerate(feature_columns):
            self.mission_database[f'{col}_norm'] = normalized[:, i]
        
        print(f"✅ Recommender trained on {len(self.mission_database)} successful missions")
        
        return self
    
    def recommend(self, current_state, n_recommendations=5):
        """
        Recommend missions similar to current state.
        
        Parameters:
        -----------
        current_state : dict
            Current UAV state {feature: value}
        n_recommendations : int
            Number of recommendations
        
        Returns:
        --------
        recommendations : list
            List of (mission, similarity_score) tuples
        """
        # Normalize current state
        current_vector = np.array([current_state.get(col, 0) for col in self.feature_columns])
        current_normalized = (current_vector - self.scaler_mean) / self.scaler_std
        
        # Calculate similarities
        sim_func = self._get_similarity_func()
        similarities = []
        
        norm_cols = [f'{col}_norm' for col in self.feature_columns]
        
        for idx, row in self.mission_database.iterrows():
            mission_vector = row[norm_cols].values
            sim = sim_func(current_normalized, mission_vector)
            similarities.append((idx, sim, row))
        
        # Sort by similarity (descending)
        similarities.sort(key=lambda x: x[1], reverse=True)
        
        return similarities[:n_recommendations]
    
    def suggest_optimal_parameters(self, current_state, n_similar=10):
        """
        Suggest optimal mission parameters based on similar successful missions.
        """
        recommendations = self.recommend(current_state, n_similar)
        
        # Aggregate recommended parameters (weighted by similarity)
        total_weight = sum([r[1] for r in recommendations])
        
        suggested = {}
        for col in ['planned_distance', 'flight_speed', 'altitude']:
            if col in self.mission_database.columns:
                weighted_sum = sum([r[2][col] * r[1] for r in recommendations])
                suggested[col] = weighted_sum / total_weight if total_weight > 0 else 0
        
        # Also suggest max safe range
        max_ranges = [r[2]['max_range'] for r in recommendations]
        suggested['recommended_max_range'] = np.percentile(max_ranges, 25)  # Conservative
        suggested['average_success_range'] = np.mean(max_ranges)
        
        return suggested


print("✅ MissionRecommender class defined!")

In [ ]:
# Train recommender
feature_cols = ['battery_soc', 'wind_speed', 'dust_level', 'payload_weight', 'power_consumption']

recommender = MissionRecommender(similarity_metric='cosine')
recommender.fit(df, feature_cols)

In [ ]:
# Test recommendation
current_state = {
    'battery_soc': 75,
    'wind_speed': 5.0,
    'dust_level': 20,
    'payload_weight': 1.0,
    'power_consumption': 500
}

print("=" * 60)
print("  CURRENT UAV STATE")
print("=" * 60)
for k, v in current_state.items():
    print(f"  {k}: {v}")

# Get recommendations
recommendations = recommender.recommend(current_state, n_recommendations=5)

print("\n" + "=" * 60)
print("  TOP 5 SIMILAR SUCCESSFUL MISSIONS")
print("=" * 60)
for i, (idx, sim, mission) in enumerate(recommendations, 1):
    print(f"\n  #{i} - Similarity: {sim:.3f}")
    print(f"      Terrain: {mission['terrain_type']}")
    print(f"      Max Range: {mission['max_range']:.2f} km")
    print(f"      Planned Distance: {mission['planned_distance']:.2f} km")

# Get optimal parameters
suggestions = recommender.suggest_optimal_parameters(current_state)

print("\n" + "=" * 60)
print("  RECOMMENDED MISSION PARAMETERS")
print("=" * 60)
for k, v in suggestions.items():
    print(f"  {k}: {v:.2f}")

In [ ]:
# Visualize similarity comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test different similarity metrics
metrics = ['cosine', 'euclidean', 'manhattan', 'pearson']
all_sims = {}

sample_missions = df[df['mission_success'] == 1].sample(100, random_state=42)
current_vec = np.array([current_state[c] for c in feature_cols])
current_norm = (current_vec - recommender.scaler_mean) / recommender.scaler_std

for metric in metrics:
    rec = MissionRecommender(similarity_metric=metric)
    rec.fit(df, feature_cols)
    
    sims = []
    sim_func = rec._get_similarity_func()
    
    for _, row in sample_missions.iterrows():
        mission_vec = np.array([row[c] for c in feature_cols])
        mission_norm = (mission_vec - rec.scaler_mean) / rec.scaler_std
        sims.append(sim_func(current_norm, mission_norm))
    
    all_sims[metric] = sims

# Plot distributions
ax = axes[0]
for metric, sims in all_sims.items():
    ax.hist(sims, bins=20, alpha=0.5, label=metric)
ax.set_xlabel('Similarity Score')
ax.set_ylabel('Frequency')
ax.set_title('Similarity Score Distributions by Metric', fontweight='bold')
ax.legend()

# Top recommendations comparison
ax = axes[1]
x = np.arange(5)
width = 0.2

for i, metric in enumerate(metrics):
    rec = MissionRecommender(similarity_metric=metric)
    rec.fit(df, feature_cols)
    recs = rec.recommend(current_state, 5)
    scores = [r[1] for r in recs]
    ax.bar(x + i*width, scores, width, label=metric)

ax.set_xlabel('Recommendation Rank')
ax.set_ylabel('Similarity Score')
ax.set_title('Top 5 Recommendations by Metric', fontweight='bold')
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(['1st', '2nd', '3rd', '4th', '5th'])
ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/recommender_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 💾 Save Models

In [ ]:
# Save recommender config
recommender_config = {
    'feature_columns': feature_cols,
    'scaler_mean': recommender.scaler_mean.tolist(),
    'scaler_std': recommender.scaler_std.tolist(),
    'similarity_metric': 'cosine',
    'n_missions': len(recommender.mission_database)
}

with open('../models/recommender_config.json', 'w') as f:
    json.dump(recommender_config, f, indent=2)

print("\n✅ Recommender configuration saved!")

---

## 📚 Learning Resources

### Time Series
1. **Moving Averages**: [Complete Guide](https://www.investopedia.com/terms/m/movingaverage.asp)
2. **Anomaly Detection**: [Time Series Anomaly Detection](https://towardsdatascience.com/time-series-anomaly-detection-with-python-60e46c7e6802)

### Recommender Systems
1. **Content-Based Filtering**: [Introduction](https://developers.google.com/machine-learning/recommendation/content-based/basics)
2. **Similarity Metrics**: [Distance Metrics Guide](https://towardsdatascience.com/9-distance-measures-in-data-science-918109d069fa)

---

## ✅ Tasks 13-16 Complete!

### What We Accomplished:
- ✅ Implemented SMA, EMA, linear trend from scratch
- ✅ Built time series forecasting
- ✅ Created anomaly detection (Z-score, IQR, Rolling)
- ✅ Implemented 4 similarity metrics from scratch
- ✅ Built content-based mission recommender
- ✅ Created mission parameter suggestions

### Classes Built:
- `TimeSeriesAnalyzer`: Moving averages, trends, forecasting
- `AnomalyDetector`: Z-score, IQR, rolling detection
- `SimilarityMetrics`: Cosine, Euclidean, Manhattan, Pearson
- `MissionRecommender`: Content-based recommendations

### Next: Streamlit Dashboard!

---